# Experiment Folder Creator
This file is for loading in a base config from ./base_configs, modifying a few of the fields, and then creating the experiment folder

In [ ]:
## Load in base config
from pathlib import Path
import yaml
import copy

base_config_path = Path("base_configs/farming_stage0.yaml")
# Load yaml file in as dictionary
with open(base_config_path, "r") as f:
    base_config = yaml.safe_load(f)

# Quick sanity display (in a notebook this will print the dict)
base_config


In [ ]:
## Add logic where you could replace e.g. the variance levels in the arms

def set_arm_std_devs(config, std_devs):
    """Return a copy of config with arm std_dev values replaced.

    - config: dict loaded from YAML
    - std_devs: list of numbers matching the order of actions in config['subtasks'][0]['params']['actions']
    """
    cfg = copy.deepcopy(config)
    actions = cfg.get("subtasks", [])[0].get("params", {}).get("actions", [])
    if len(std_devs) != len(actions):
        raise ValueError("std_devs length must match number of actions")
    for a, s in zip(actions, std_devs):
        a["std_dev"] = s
    return cfg


def generate_variants(base_cfg, std_dev_grid):
    """Generate configs for each std_dev setting in std_dev_grid.

    - std_dev_grid: iterable of lists, each inner list is std_devs for all arms
    Returns list of tuples: (name_suffix, cfg)
    """
    variants = []
    for i, sd in enumerate(std_dev_grid):
        name = f"var_{i}"
        cfg = set_arm_std_devs(base_cfg, sd)
        variants.append((name, cfg))
    return variants

# Example: create two variants (low and high variance for arm 3)
std_dev_grid = [
    [0, 0, 0],    # all deterministic
    [0, 0, 10],   # make Field_3 noisy
]
variants = generate_variants(base_config, std_dev_grid)
variants


In [ ]:
## Create a folder with an identifier passed by the user and the date and time. Save the config there
import datetime
import json


def save_variants(variants, experiments_dir="experiments", run_id=None):
    """Save each variant YAML into a timestamped experiments subfolder.

    Returns the path to the experiments run folder created.
    """
    base = Path(experiments_dir)
    base.mkdir(parents=True, exist_ok=True)
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    run_id = run_id or f"run_{ts}"
    run_path = base / run_id
    run_path.mkdir(parents=True, exist_ok=False)

    manifest = []
    for name, cfg in variants:
        fname = f"{name}.yaml"
        out_path = run_path / fname
        with open(out_path, "w") as f:
            yaml.safe_dump(cfg, f)
        manifest.append({"name": name, "path": str(out_path)})

    # Save a run manifest (json)
    with open(run_path / "manifest.json", "w") as f:
        json.dump({"created_at": ts, "variants": manifest}, f, indent=2)

    return run_path

# Example usage (uncomment to run in notebook):
# run_path = save_variants(variants, experiments_dir="experiments", run_id=None)
# run_path
